# Generative Adversarial Network (GAN) - PyTorch

**Goal:** Train a small generator/discriminator pair on digits.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** A generator and discriminator improve through adversarial feedback.
- **Where it is used:** image synthesis, augmentation research, and generator-discriminator demos.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Generative Adversarial Network: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['noise', 'generator', 'judge']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = 1/(1+np.exp(-x))
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.48,.52])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['real', 'fake'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
from sklearn.datasets import load_digits

X = (load_digits().data / 16.0).astype("float32")
train_loader = DataLoader(TensorDataset(torch.tensor(X)), batch_size=64, shuffle=True, drop_last=True)
latent_dim = 32


In [ ]:
generator = nn.Sequential(
    nn.Linear(latent_dim, 64),
    nn.LeakyReLU(0.2),
    nn.Linear(64, 128),
    nn.LeakyReLU(0.2),
    nn.Linear(128, 64),
    nn.Sigmoid(),
).to(device)

discriminator = nn.Sequential(
    nn.Linear(64, 128),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.2),
    nn.Linear(128, 1),
).to(device)

opt_g = torch.optim.AdamW(generator.parameters(), lr=2e-4)
opt_d = torch.optim.AdamW(discriminator.parameters(), lr=2e-4)
criterion = nn.BCEWithLogitsLoss()


In [ ]:
for epoch in range(20):
    for (real,) in train_loader:
        real = real.to(device)
        batch_size = real.size(0)

        noise = torch.randn(batch_size, latent_dim, device=device)
        fake = generator(noise).detach()
        real_loss = criterion(discriminator(real), torch.ones(batch_size, 1, device=device))
        fake_loss = criterion(discriminator(fake), torch.zeros(batch_size, 1, device=device))
        d_loss = real_loss + fake_loss
        opt_d.zero_grad(set_to_none=True)
        d_loss.backward()
        opt_d.step()

        noise = torch.randn(batch_size, latent_dim, device=device)
        generated = generator(noise)
        g_loss = criterion(discriminator(generated), torch.ones(batch_size, 1, device=device))
        opt_g.zero_grad(set_to_none=True)
        g_loss.backward()
        opt_g.step()

    print(f"epoch={epoch+1:02d} d_loss={d_loss.item():.3f} g_loss={g_loss.item():.3f}")
